# Forward-scan sonar: what the image knows and what it loses

This is an undergraduate learning simulator with synthetic validation, built with AI coding assistance. It is not a calibrated DIDSON or a reproduction of a complete research paper.

Sound is a pressure disturbance traveling through water. A sonar sends sound and listens for an echo. If sound speed is **c**, elapsed time is **t**, and distance is **r**, then **r = ct/2**. The division by two accounts for the outward and return trips. Here c is usually 1500 metres per second.

A bearing is a left-right direction. We write it as **theta (θ)**. Elevation is an up-down angle, written **phi (φ)**. Range **r** and bearing **θ** label the sonar pixel; elevation **φ** does not. Many 3-D points share that pixel.

The coordinate frame has X to starboard/right, Y forward, and Z up. **P** means a point, represented by its three coordinates. The equation is

$$P=r[\cos(\phi)\sin(\theta),\cos(\phi)\cos(\theta),\sin(\phi)].$$

Cosine and sine convert angles to coordinate fractions. A rotation expresses the same physical point along different sensor axes. It does not remove elevation; discarding φ when choosing a pixel does that.


In [ ]:
import sys, glob, os
from pathlib import Path
if not Path("setup.py").exists() and Path("../setup.py").exists():
    os.chdir("..")
sys.path.insert(0, "."); sys.path.insert(0, "python")

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, Audio, display

import acoustics, montecarlo, outputs, recovery, scene as sc, sonar


# One run supplies every artifact. Never mix figures from different parameter states.
from pathlib import Path
root = Path.cwd()
if not (root / 'setup.py').exists() and (root.parent / 'setup.py').exists():
    os.chdir(root.parent)
    sys.path[:0] = ['.', 'python']
selected = os.environ.get('SONAR_OUTPUT_DIR')
if selected is None:
    complete = [p for p in sorted(Path('results').glob('*'), reverse=True)
                if (p / 'demo17_metrics.json').exists()]
    if not complete:
        raise RuntimeError('Run ./run_all.sh from the project root first.')
    selected = str(complete[0])

def figure(name):
    path = Path(selected) / name
    if not path.exists():
        raise FileNotFoundError(f'{path}: run the matching demonstration first')
    return str(path)
print('Figures from:', selected)
print('Cython extension:', sonar.__file__)


## 1. The ambiguity is a property of the model, not an approximation

Put a target at range `r`, bearing `theta`, elevation `+phi`. Put an identical target
at `(r, theta, -phi)`. Those are two genuinely different places in the water. Render
both.

The image is addressed by range and bearing, and the vertical beam weighting `B(phi)`
is even in `phi`, so nothing in the pipeline can tell them apart. If a bug leaked
elevation into a pixel address, this test would fail.

In [ ]:
sim = sonar.SonarSimulator(frequency_hz=1.2e6, num_azimuth_bins=193, num_range_bins=700,
                           horizontal_fov_deg=30.0, vertical_beamwidth_deg=14.0,
                           max_range_m=10.0, num_elevation_subrays=1024)

pair = [sim.render([sonar.make_sphere(sc.spherical_to_world(6.0, 4.0, sign * 5.0), 0.15, 0.9)])
        for sign in (+1, -1)]
difference = float(np.abs(pair[0] - pair[1]).max())
print(f"peak intensity        {pair[0].max():.6e}")
print(f"max |difference|      {difference:.3e}")
print(f"relative to peak      {difference / pair[0].max():.3e}")

The residual is not zero only because the sub-ray that strikes the `+phi` target is
`s` while for `-phi` it is `n-1-s`, so the contributions are summed in the opposite
order and floating-point addition is not associative. That is double-precision
round-off, not a physical difference.

In [ ]:
display(Image(filename=figure("demo1_ambiguity.png"), width=900))

## 2. What a second viewpoint can and cannot do

Translating the sonar sideways does **not** separate `+phi` from `-phi`: only `z^2`
enters the range, so the two remain identical however many pings you collect along a
constant-depth line. Translating vertically does separate them, immediately.

In [ ]:
display(Image(filename=figure("demo5_motion.png"), width=900))
print(open(figure("demo5_motion.txt")).read())

## 3. Recovering elevation with a camera

A sonar bin is consistent with a one-parameter family of 3-D points: the arc swept by
elevation across the vertical beam. A camera pixel is the complement, two angles and
no range. Project the arc into the camera, cross it with the pixel the target was
detected in, and the point is determined.

Both detections below are made off rendered images, not read out of the scene.

In [ ]:
display(Image(filename=figure("demo8_optiacoustic.png"), width=1000))

One bin alone leaves 0.959 m of arc open. Fused with the camera the elevation comes
back at 3.486 degrees against a true 3.500, a 2.4 cm position error.

The turbidity sweep in the bottom right is the other half of the argument: the optical
direct term decays as `exp(-2 c r)` while the veiling glow grows as `1 - exp(-c r)`,
so past about `c = 0.2` per metre the camera detection is gone and the pair falls back
to the sonar's arc. The sonar image itself is unchanged throughout.

### That was one run. Here it is as a distribution.

Speckle has contrast 1, so a single realisation can land anywhere and a single number
is an anecdote. Below, the same recovery is rerun with an independent speckle seed and
independent camera read noise each time, calling exactly the functions the single run
called. The interval is a percentile bootstrap rather than a t interval, because the
error comes from a centroid on a thresholded blob and is not obviously Gaussian.


In [ ]:
import montecarlo, recovery

TRIALS = 60          # the full run in demo 13 uses 200; 60 keeps the notebook quick
CONFIDENCE = 0.95

axes_mc = recovery.scene_axes()
objects_mc = recovery.build_objects(recovery.target_centre(axes_mc))
sim_mc = recovery.build_sonar()

errors = [recovery.trial(sim_mc, objects_mc, axes_mc, 9000 + i, [0.30])[1][0]
          - recovery.TARGET_ELEVATION for i in range(TRIALS)]
stats = montecarlo.summarise(errors, CONFIDENCE, seed=1)
print(montecarlo.format_summary("elevation error", stats, "degrees"))
print(f"elevation error: {stats['mean']:+.4f} plus or minus {stats['std']:.4f} "
      f"degrees across {TRIALS} trials")


The full 200-trial run, with the camera-baseline sweep and the same treatment applied
to the texture-versus-speckle correlation and to the chirp's range error:


In [ ]:
display(Image(filename=figure("demo13_montecarlo.png"), width=1000))


Panel B is the result worth pausing on. The recovery's bias grows steadily with the
camera's mounting baseline, from essentially zero at 5 cm to about -0.047 degrees at
60 cm. A co-located camera reads elevation straight off the pixel and the sonar's range
error does not enter; the further the camera sits from the head, the more that range
error projects into the recovered angle. Every interval here is tight enough that the
trend is not noise.


## 4. The acoustic chain: chirp, matched filter, range resolution

Everything above forms an image geometrically. This section works in the time domain
instead: a transmitted waveform, echoes delayed by `tau = 2r/c`, noise, and a matched
filter.

A plain pulse of duration `T` cannot separate echoes closer than `c T / 2`. A linear FM
chirp sweeping `B` hertz over the same `T` compresses in the matched filter to a pulse
of width about `1/B`, so the resolution becomes `c / (2B)`, independent of `T`. The
improvement is the time-bandwidth product `B T`.

In [ ]:
chirp_sonar = sonar.ChirpSonar(frequency_hz=300e3, chirp_bandwidth_hz=60e3,
                               chirp_duration_s=2e-3, sample_rate_hz=1.2e6)
print(f"time-bandwidth product BT   {chirp_sonar.time_bandwidth_product:.0f}")
print(f"chirp resolution  c/(2B)    {1000 * chirp_sonar.range_resolution_m:.2f} mm")
print(f"plain pulse       cT/2      {chirp_sonar.uncompressed_resolution_m:.3f} m")

# One reflector, no noise: measure the compressed width against theory.
sweep = chirp_sonar.transmit(swept=True)
solo = acoustics.envelope(
    chirp_sonar.compress(chirp_sonar.receive([(4.0, 1.0, 1.0)], sweep, 0.010), sweep))
ranges = chirp_sonar.range_axis_m(len(solo))
width = acoustics.half_power_width(solo, int(np.argmax(solo))) * (ranges[1] - ranges[0])
print(f"\nmeasured half-power width   {1000 * width:.2f} mm")
print(f"theory 0.886 c/(2B)         {1000 * 0.886 * chirp_sonar.range_resolution_m:.2f} mm")

In [ ]:
display(Image(filename=figure("demo12_chirp.png"), width=1000))

Two targets 50 mm apart come back at -0.62 mm and +0.63 mm, inside one sample. A plain
pulse of the same duration merges them into a single blob.

## 5. Listening to it

The three waveforms below are the real records, resampled onto a time base 600 times
longer. That divides every frequency by 600 and leaves the shape untouched, so the
300 kHz carrier plays at 500 Hz.

In [ ]:
for label, caption in (("transmit", "transmitted chirp, 300 to 360 kHz sweep"),
                       ("received", "received record: two echoes buried in noise"),
                       ("compressed", "after the matched filter: a compressed click")):
    path = figure(f"demo12_chirp_{label}.wav")
    print(caption)
    display(Audio(filename=path))

## Where this stands

Reproduced end to end: the elevation ambiguity to 1.1e-15 of peak, the intensity model
against its closed form to 2.3e-16 relative, shadow-based height recovery to 0.93 mm
rms about a fitted line, opti-acoustic elevation recovery to 0.11 cm of height error
over 200 trials, and pulse compression to 0.2% of the theoretical width.

Not yet built, and deliberately not faked here: delay-and-sum digital beamforming from
the raw element signals, sound-speed refraction, a CFAR detector on the noisy images,
and a transducer response model for the transmitted waveform.


## Meshes and elevation integration

A triangle is a small flat patch. Its normal is an arrow perpendicular to it. OBJ files store vertices and the triples that join them. The C++ loader converts explicit units to metres, loads material coefficients and intersects these triangles using barycentric coordinates (weights describing where a point lies inside a triangle).

For every horizontal bearing the renderer samples many elevations, keeps each ray's first hit, weights the return by the vertical beam and deposits it into a range bin. It averages the contributions over equally spaced elevation samples. The normalized midpoint average approximates an angular mean; it is not calibrated received watts. `legacy_elevation_sum=True` retains the previous endpoint sum for historical reproduction.

The orange overlay below is projected triangle wireframe. Elevation is visibly absent from the bottom row's axes.

In [ ]:
display(Image(filename=figure('demo14_meshes.png'), width=1100))

## Why highlights weaken with distance

**R** is an assumed material reflectivity, **n** a unit surface normal, and **u** the outgoing unit ray. The dot product **n·(-u)** measures how directly the surface faces the sonar: one for facing it, zero for grazing. A highlight is a visible surface contributing a relatively strong return.

$$I=R\max(0,n\cdot(-u))r^{-4}10^{-TL/10}.$$

**I** is relative linear intensity. The fourth power combines outward and return spherical spreading for a point-scatterer approximation. An extended calibrated target need not follow the same simple law.

**α (alpha)** is absorption in decibels per kilometre. In Thorp's empirical formula, **f** is frequency in kilohertz:

$$\alpha=0.11f^2/(1+f^2)+44f^2/(4100+f^2)+2.75\times10^{-4}f^2+0.003.$$

**TL** is two-way absorption loss in decibels: **TL = 2αr/1000** for r in metres. This omits source level, receiver calibration, system noise and many terms in a complete sonar equation. Extrapolating Thorp to MHz does not establish calibrated seawater accuracy.

Raw values drive reconstruction. Log compression and green display normalization are separate copies.

In [ ]:
display(Image(filename=figure('demo3_validation.png'), width=1000))

## Beam shape and shadows

**λ (lambda)** means wavelength, c divided by frequency. **N** is array element count and **d** is spacing. The uniform array's power is

$$B(\phi)=\left[\frac{\sin(N\pi d\sin\phi/\lambda)}{N\sin(\pi d\sin\phi/\lambda)}\right]^2.$$

At zero angle the limit is one. Hann shading gradually weakens edge elements, suppressing sidelobes but broadening the main beam. Applying this ideal array along elevation is an explicit approximation.

A first hit blocks a farther hit on the same ray. Some elevations can see the seabed while others are blocked, producing a partially visible bin. The complete configured range is plotted below. Sample-count convergence is measured rather than inferred from appearance.

In [ ]:
display(Image(filename=figure('demo14_beams.png'), width=800))
display(Image(filename=figure('demo2_shadow.png'), width=1100))
display(Image(filename=figure('demo14_convergence.png'), width=800))

## Several poses and the feasible volume

A **voxel** is a small cube. Start with a grid, project every cube centre into each image and remove a cube if an observed projection is inconsistent with highlight or feasible shadow. A cube outside a view is retained because that view provides no evidence about it. The mask here conservatively allows farther cells on a bearing with a highlight. It does not claim to measure a physical shadow from darkness alone.

**IoU** is overlapping predicted/true volume divided by their union. One is perfect agreement. **False positives** are retained cubes outside the true object; **false negatives** are true cubes removed. Normalized volume error is (predicted volume − true volume)/true volume.

The result is a sonar visual hull, or feasible volume. Hidden concavities and missing poses remain uncertain. Known poses and synthetic truth make this easier than real data.

In [ ]:
display(Image(filename=figure('demo15_carving.png'), width=1100))
import json
carving=json.load(open(figure('demo15_metrics.json')))
carving['poses']

## Correlated speckle and reconstruction error

Many unresolved scatterers have random phases. Summing their complex contributions can give Gaussian real and imaginary parts. Their magnitude has a Rayleigh distribution; its square has an exponential distribution. Single-look intensity speckle is simulated as **I observed = I true × (−ln U)**, with U uniform between zero and one.

**σ (sigma)** usually means standard deviation here. Speckle contrast is standard deviation divided by mean, near one for a fully developed single look. **1 − exp(−0.1) = 9.52%** of samples fall below one tenth of the mean.

The correlated mode filters a complex Gaussian field, then squares its magnitude. Correlation length is in pixel cells, and FFT filtering wraps at image boundaries. No frame-by-frame normalization is used to force the statistics. The optional contrast mixture and multi-look average are no longer single-look exponential distributions.

Finite beam size, bandwidth, pulse compression and interpolation can correlate real cells. Deterministic texture is separately attached to a target; random speckle is redrawn per ping. More looks reduce contrast, but a threshold tied to each image's random maximum can make reconstruction trends nonmonotonic. The curves are evidence for this segmentation rule, not a universal law.

Each error bar below is a bootstrap 95% interval for the mean over 20 random-ping trials. It is not the range containing 95% of individual reconstructions.

In [ ]:
display(Image(filename=figure('demo4_speckle.png'), width=1000))
display(Image(filename=figure('demo15_degradation.png'), width=1200))

## Point recovery and conditioning

The **Jacobian** is a small table: each entry says how much a range or bearing changes when a point moves a little along X, Y or Z. Its **rank** counts locally observable motion directions. **Singular values** measure sensitivity along three independent directions; a very small value means noise is amplified.

Sideways coplanar poses retain an above/below reflection ambiguity. At zero elevation their vertical sensitivity vanishes. Vertical movement can break that ambiguity. The experiment uses known correspondences and starts on the positive-elevation branch. Pose noise is 2 mm and 0.05 degrees; measurement noise is 3 mm range and 0.05 degrees bearing. Failure means position error exceeds 10 cm. These choices are recorded, not fitted to a desired outcome.

In [ ]:
display(Image(filename=figure('demo16_point_recovery.png'), width=1000))
json.load(open(figure('demo16_metrics.json')))

## Sea-surface returns

Mirror the sonar position across a flat water surface. **r_d** is direct distance, **r_m** is reflected-leg distance, and **r_g = (r_d+r_m)/2** is a mixed-path apparent range. The mirror component uses its own projected bearing. Ghosts are not discarded solely because a mirror leaves the beam.

**σ_h** means RMS surface height, **g** grazing angle, and **k=2π/λ** wavenumber. The simplified coherence amplitude is **γ=exp[−(2kσ_h sin g)²/2]**. Received ghost power uses γ squared; two-bounce mirror power uses γ to the fourth power. Lost coherent energy is not converted into diffuse reverberation.

All three components are seeded by direct-visible patches; reflected-leg occlusion is absent. Generating these approximations is easier than identifying and removing overlapping artifacts from real observations. No ghost removal is implemented.

In [ ]:
display(Image(filename=figure('demo17_components.png'), width=1200))
display(Image(filename=figure('demo17_roughness.png'), width=800))

## Four models and the remaining research gap

The one-ray mode omits elevation accumulation and can miss off-axis patches or misrepresent shadows. The comparison uses a common display reference and saves timing configuration. The full row selects correlated speckle and roughness; it does not add unmodeled physics.

The 2024 paper's complete patch-motion objective is not reproduced. No ICP/IRLS contour-to-patch motion pipeline, mirror-motion consistency objective, depth/tilt optimization or iterative mesh refinement is claimed. See `docs/paper_mapping.md` for verified metadata and source-access limitations.

In [ ]:
display(Image(filename=figure('demo14_renderers.png'), width=1200))

## Limits and honest authorship

The array is idealized; scattering is diffuse; material reflectivity is simplified; r⁻⁴ is a point-scatterer assumption. Absorption is empirical and the sonar equation incomplete. Sea-surface geometry and coherence are approximations. There is no complete concavity reverberation or calibrated DIDSON response. Speckle is independent unless correlation is enabled. Segmentation and known poses control carving. Synthetic validation is not real-data validation. A feasible hull is not unique geometry.

I used AI as a coding assistant while testing the models and learning the underlying physics. I can distinguish analytic agreement, numerical experiment results, and unvalidated assumptions. I do not claim eight months of work, sole authorship without assistance, institutional endorsement, or research-grade accuracy.